# GCN 한 층 직접 짜기

GCN 한 층을 세 번 짜요. 먼저 브라우저에서 numpy 로 A + I, 차수, 정규화, 곱하기, ReLU 를 한 줄씩. 다음 Colab 에서 torch nn.Module 로. 마지막엔 PyG 의 GCNConv 를 쓰고, 2주차 노트북의 ScratchGCN 과 학습 함수를 작은 가짜 그래프로 혼자 다시 짜요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 준비: 필요한 도구 설치 (처음 한 번만 실행)

In [ ]:
!pip install -q torch_geometric

## 1. torch 로 정규화 행렬 A_hat 만들기  (따라 치기)

c12-01 ~ c12-03 에서 numpy 로 만든 A_hat 을 torch 텐서로 똑같이 만들어요.

- 짝꿍 numpy 실습은 c12-02, c12-03 이에요. 바뀌는 것은 이름뿐이에요: `np.eye` 는 `torch.eye`, `axis` 는 `dim`, `np.diag` 는 `torch.diag`(c09).
- `deg.pow(-0.5)` 는 `deg ** -0.5` 와 같아요. 수업 노트북이 pow 를 써서 여기서도 pow 로 적었어요.
- `dtype=torch.float32` 를 꼭 붙여요. 정수 텐서로 두면 나중에 소수 가중치와 곱할 때 dtype 에러가 나요(c09-06).
- torch 로 만들어야 뒤에서 학습(자동 미분)과 이어 쓸 수 있어요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch

A = torch.tensor([[0, 1, 1, 0],
                  [1, 0, 1, 0],
                  [1, 1, 0, 1],
                  [0, 0, 1, 0]], dtype=torch.float32)
n = A.shape[0]
A_tilde = A + torch.eye(n)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
print(A_hat)
```

<details><summary>힌트 1</summary>

c12-03 numpy 코드를 옆에 두고 한 줄씩 바꿔요

</details>

<details><summary>힌트 2</summary>

sum 에는 dim=1

</details>

<details><summary>힌트 3</summary>

보이는 코드를 그대로 쳐요

</details>


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
_A4 = torch.tensor([[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]], dtype=torch.float32)
for _name in ['A', 'A_tilde', 'deg', 'D_inv_sqrt', 'A_hat']:
    assert _name in globals(), f"{_name} 을 만들어야 해요"
assert torch.is_tensor(A) and A.dtype == torch.float32, "A 는 dtype=torch.float32 텐서여야 해요"
assert torch.allclose(deg, torch.tensor([3.0, 3.0, 4.0, 2.0])), f"deg 는 [3, 3, 4, 2] 여야 해요. 지금은 {deg.tolist()} 예요. A_tilde.sum(dim=1)"
assert tuple(A_hat.shape) == (4, 4), f"A_hat 모양은 (4, 4) 여야 해요. 지금은 {tuple(A_hat.shape)} 예요"
assert torch.allclose(A_hat, _norm_t(_A4), atol=1e-6), "A_hat 값이 달라요. D_inv_sqrt @ A_tilde @ D_inv_sqrt 순서와 @ 를 확인해요"
assert abs(A_hat[0, 1].item() - 1 / 3) < 1e-6, "A_hat[0, 1] 은 0.333 이어야 해요"
print("통과! 2주차 PracticeCode_2 셀 29 도 torch 로 A_hat 을 만들어요. 다만 인접행렬 대신 엣지 목록에서 칸을 하나씩 채워요.")

<details><summary>정답 보기</summary>

```python
import torch

A = torch.tensor([[0, 1, 1, 0],
                  [1, 0, 1, 0],
                  [1, 1, 0, 1],
                  [0, 0, 1, 0]], dtype=torch.float32)
n = A.shape[0]
A_tilde = A + torch.eye(n)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
print(A_hat)
```

</details>

## 2. GCNLayer 클래스: 한 층을 nn.Module 로  (따라 치기)

c12-05 의 GCN 한 층을 nn.Module 클래스로 만들어요. 가중치 W 는 nn.Linear 가 들고 있어서 학습할 수 있어요.

- 짝꿍 numpy 실습은 c12-04, c12-05 예요. 클래스 모양은 c10 의 MLP 와 같아요: `nn.Module` 물려받기, `super().__init__()`, 층을 속성으로, forward.
- `nn.Linear(in_dim, out_dim, bias=False)` 는 더하는 숫자(bias) 없이 가중치 곱하기만 해요. 식의 W 가 이것이에요.
- `self.W(H)` 는 `H @ W` 를 계산해요. 그래서 `A_hat @ self.W(H)` 가 식 $\hat{A} H W$ 예요. (nn.Linear 속 weight 는 (출력, 입력) 모양이라 속에서는 `H @ weight.T` 로 곱해요.)
- `activation=True` 는 기본값이에요(c04). 안 적으면 True, `activation=False` 로 부르면 ReLU 를 건너뛰어요.
- `torch.relu(out) if activation else out` 은 한 줄짜리 if 예요. "activation 이 참이면 torch.relu(out), 아니면 out" 이라고 읽어요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

layer = GCNLayer(3, 2)
```

<details><summary>힌트 1</summary>

c10 의 MLP 클래스 모양을 떠올려요

</details>

<details><summary>힌트 2</summary>

forward 의 매개변수 순서는 H, A_hat, activation

</details>

<details><summary>힌트 3</summary>

보이는 코드를 그대로 쳐요

</details>


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
assert 'GCNLayer' in globals() and issubclass(GCNLayer, nn.Module), "GCNLayer 는 nn.Module 을 물려받는 클래스여야 해요"
assert 'layer' in globals() and isinstance(layer, GCNLayer), "layer = GCNLayer(3, 2) 를 만들어요"
assert isinstance(layer.W, nn.Linear), "self.W = nn.Linear(in_dim, out_dim, bias=False) 가 있어야 해요"
assert layer.W.bias is None, "GCN 한 층의 W 에는 bias 가 없어요. bias=False 를 붙여요"
assert tuple(layer.W.weight.shape) == (2, 3), f"GCNLayer(3, 2) 의 weight 모양은 (2, 3) 이어야 해요. 지금은 {tuple(layer.W.weight.shape)} 예요"
torch.manual_seed(1)
_H = torch.randn(4, 3)
_Ah = _norm_t(torch.tensor([[0, 1, 1, 0], [1, 0, 1, 0], [1, 1, 0, 1], [0, 0, 1, 0]], dtype=torch.float32))
with torch.no_grad():
    _raw = _Ah @ (_H @ layer.W.weight.T)
    _o1 = layer(_H, _Ah)
    _o2 = layer(_H, _Ah, activation=False)
assert tuple(_o1.shape) == (4, 2), f"노드 4개를 넣으면 결과 모양은 (4, 2) 여야 해요. 지금은 {tuple(_o1.shape)} 예요"
assert torch.allclose(_o1, torch.relu(_raw), atol=1e-6), "layer(H, A_hat) 는 relu(A_hat @ W(H)) 여야 해요"
assert torch.allclose(_o2, _raw, atol=1e-6), "activation=False 면 ReLU 없이 A_hat @ W(H) 를 돌려줘야 해요"
_big = GCNLayer(5, 4)
assert tuple(_big.W.weight.shape) == (4, 5), "GCNLayer(5, 4) 의 weight 모양은 (4, 5) 여야 해요. in_dim, out_dim 을 nn.Linear 에 그대로 넘겨요"
print("통과! 2주차 PracticeCode_2 셀 55 의 `GCNLayer` 가 이 클래스예요. 노트북에는 첫 줄에 설명 문자열 한 줄이 더 있어요(c12-13 에서 그대로 짜요).")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

layer = GCNLayer(3, 2)
```

</details>

## 3. 고치기: A_hat 을 뒤에 곱해서 모양 에러  (고치기)

GCNLayer 를 불렀더니 `RuntimeError: mat1 and mat2 shapes cannot be multiplied` 가 나요. forward 의 곱하는 순서를 고쳐요.

- 짝꿍은 c09-10 의 모양 에러예요. 행렬곱은 **순서를 바꾸면 안 돼요**.
- `self.W(H)` 의 모양은 (노드 4, 새 특징 2) 예요. `(4, 2) @ A_hat(4, 4)` 는 안쪽 숫자 2 와 4 가 달라서 곱할 수 없어요.
- `A_hat @ self.W(H)` 는 (4, 4) @ (4, 2) = (4, 2) 예요. 식 $\hat{A} H W$ 에서도 A_hat 이 맨 앞이에요.
- 에러가 나는 건 정상이에요. 메시지 마지막 줄의 모양 숫자부터 읽어요.

<details><summary>힌트 1</summary>

에러 난 줄은 forward 안의 out = ... 줄이에요

</details>

<details><summary>힌트 2</summary>

식 A_hat H W 에서 A_hat 은 어디에 있죠?

</details>

<details><summary>힌트 3</summary>

out = A_hat @ self.W(H)

</details>


In [ ]:
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = self.W(H) @ A_hat
        return torch.relu(out) if activation else out

torch.manual_seed(0)
A_hat = torch.full((4, 4), 0.25)   # 모두가 1/4 씩 섞이는 가짜 A_hat
H = torch.rand(4, 3)
layer = GCNLayer(3, 2)
out = layer(H, A_hat)
print(out.shape)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
assert 'out' in globals() and torch.is_tensor(out), "out = layer(H, A_hat) 가 있어야 해요"
assert tuple(out.shape) == (4, 2), f"out 모양은 (4, 2) 여야 해요. 지금은 {tuple(out.shape)} 예요"
_L = GCNLayer(3, 5)
torch.manual_seed(3)
_H = torch.randn(6, 3)
try:
    with torch.no_grad():
        _o = _L(_H, _Ahat6, activation=False)
except RuntimeError as _e:
    raise AssertionError(f"노드 6개, 새 특징 5개에서 모양 에러가 나요: {_e}. out = A_hat @ self.W(H) 순서로 곱해요")
with torch.no_grad():
    _want = _Ahat6 @ (_H @ _L.W.weight.T)
assert tuple(_o.shape) == (6, 5), f"노드 6개, 새 특징 5개면 모양은 (6, 5) 여야 해요. 지금은 {tuple(_o.shape)} 예요"
assert torch.allclose(_o, _want, atol=1e-6), "값이 달라요. A_hat 이 맨 앞, 그다음 self.W(H) 예요"
print("통과! 노드 수와 특징 수가 우연히 같으면 순서를 바꿔도 에러 없이 틀린 값이 나와요. 그래서 모양을 적어 두는 습관이 중요해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

torch.manual_seed(0)
A_hat = torch.full((4, 4), 0.25)   # 모두가 1/4 씩 섞이는 가짜 A_hat
H = torch.rand(4, 3)
layer = GCNLayer(3, 2)
out = layer(H, A_hat)
print(out.shape)
```

</details>

## 4. 빈칸: 층 두 개를 쌓은 ScratchGCN  (빈칸 채우기)

GCNLayer 두 개를 속성으로 달아 2층 GCN 모델을 만들어요. 첫 층은 ReLU 를 켜고, 마지막 층은 끄고 반별 점수를 내요.

- 층을 속성으로 다는 것은 c10 과 같아요. 첫 층은 입력 특징 수 → 숨은 특징 수(hidden_dim), 둘째 층은 숨은 특징 수 → 반 수(num_classes) 예요.
- 앞 층의 출력 수와 뒤 층의 입력 수가 같아야 이어져요. 그래서 둘째 칸에도 hidden_dim 이 들어가요.
- 마지막 층은 ReLU 를 끄는 게 약속이에요. 반별 점수(로짓)는 음수도 될 수 있어야 F.cross_entropy 가 제대로 채점해요(c11).
- `return_embeddings=True` 로 부르면 첫 층 결과 h (노드마다 숨은 특징, **임베딩(Embedding)**) 를 바로 돌려줘요. 노트북은 이걸 그림으로 그려요.
- 여기서 A_hat 은 모든 칸이 1/4 인 가짜 표예요. 뒤에서 진짜 가짜 그래프로 바꿔요.

<details><summary>힌트 1</summary>

층 사이에 흐르는 숫자 개수는 hidden_dim

</details>

<details><summary>힌트 2</summary>

첫 층은 ReLU 켬(True), 마지막 층은 끔(False)

</details>

<details><summary>힌트 3</summary>

hidden_dim, hidden_dim, True, False

</details>


In [ ]:
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, ___)
        self.conv2 = GCNLayer(___, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=___)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=___)

torch.manual_seed(42)
model = ScratchGCN(4, 8, 2)
logits = model(torch.eye(4), torch.full((4, 4), 0.25))
print(logits.shape)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
assert 'ScratchGCN' in globals() and issubclass(ScratchGCN, nn.Module), "ScratchGCN 클래스가 있어야 해요"
assert 'logits' in globals() and tuple(logits.shape) == (4, 2), "logits 모양은 (노드 4, 반 2) = (4, 2) 여야 해요"
_m = ScratchGCN(6, 5, 3)
assert tuple(_m.conv1.W.weight.shape) == (5, 6), f"ScratchGCN(6, 5, 3) 의 conv1 은 6 → 5 여야 해요. 지금 weight 모양 {tuple(_m.conv1.W.weight.shape)}. 첫 빈칸은 hidden_dim"
assert tuple(_m.conv2.W.weight.shape) == (3, 5), f"conv2 는 5 → 3 이어야 해요. 지금 weight 모양 {tuple(_m.conv2.W.weight.shape)}. 둘째 빈칸도 hidden_dim"
torch.manual_seed(7)
_x = torch.randn(6, 6)
with torch.no_grad():
    _h = torch.relu(_Ahat6 @ _m.conv1.W(_x))
    _want = _Ahat6 @ _m.conv2.W(_h)
    _emb = _m(_x, _Ahat6, return_embeddings=True)
    _out = _m(_x, _Ahat6)
assert torch.allclose(_emb, _h, atol=1e-6), "return_embeddings=True 일 때 첫 층 결과가 달라요. 첫 층은 activation=True 예요"
assert (_want < 0).any(), "채점용 입력 확인"
assert torch.allclose(_out, _want, atol=1e-6), "마지막 층 결과가 달라요. 마지막 층은 activation=False 라서 음수 점수도 남아야 해요"
print("통과! 2주차 PracticeCode_2 셀 55 의 `ScratchGCN` 이 이 클래스예요. 노트북은 34 → 16 → 2 로 써요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

torch.manual_seed(42)
model = ScratchGCN(4, 8, 2)
logits = model(torch.eye(4), torch.full((4, 4), 0.25))
print(logits.shape)
```

</details>

## 5. PyG Data: 그래프를 상자 하나에 담기  (따라 치기)

노드 6개짜리 가짜 그래프의 특징, 엣지, 정답, 정답 아는 노드 표시를 PyG 의 Data 상자 하나에 담아요.

- **PyG(PyTorch Geometric)** 는 GNN 에 필요한 것을 모아 둔 torch 추가 도구예요. Colab 노트북 맨 앞에 설치 칸이 자동으로 붙어요.
- 수업 노트북은 가라테 클럽 34명 그래프를 내려받아 쓰지만, 여기서는 내려받기 없이 돌도록 **노드 6개짜리 가짜 그래프** 로 바꿨어요. 0-1-2 삼각형과 3-4-5 삼각형이 2-3 선 하나로 이어져 있고, 앞 3명은 0반, 뒤 3명은 1반, 정답을 아는 노드는 0번과 5번뿐이에요.
- `edge_index` 는 2줄짜리 표예요. 위 줄이 출발 노드, 아래 줄이 도착 노드이고, 무방향이라 선 하나를 두 방향으로 두 번 적어요(c08). 선 7개라 칸이 14개예요.
- `Data(x=..., edge_index=..., y=...)` 는 이름 붙은 칸이 있는 상자예요. `data.x`, `data.edge_index`, `data.y` 로 꺼내요.
- `data.train_mask` 처럼 상자에 새 칸을 나중에 붙일 수도 있어요. dtype=torch.bool 은 True/False 표예요(c11).

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
from torch_geometric.data import Data

# 가짜 그래프: 0-1-2 삼각형, 3-4-5 삼각형, 2-3 다리
edge_index = torch.tensor([[0, 1, 0, 2, 1, 2, 2, 3, 3, 4, 3, 5, 4, 5],
                           [1, 0, 2, 0, 2, 1, 3, 2, 4, 3, 5, 3, 5, 4]], dtype=torch.long)
x = torch.eye(6)
y = torch.tensor([0, 0, 0, 1, 1, 1])
data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = torch.zeros(6, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[5] = True
print(data, data.num_nodes, data.num_edges)
```

<details><summary>힌트 1</summary>

edge_index 두 줄의 숫자를 한 칸씩 짝지어 보면 (0, 1), (1, 0), ... 이에요

</details>

<details><summary>힌트 2</summary>

Data 는 from torch_geometric.data import Data

</details>

<details><summary>힌트 3</summary>

보이는 코드를 그대로 쳐요

</details>


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
from torch_geometric.data import Data as _Data
assert 'data' in globals() and isinstance(data, _Data), "data = Data(x=x, edge_index=edge_index, y=y) 를 만들어요"
assert tuple(data.x.shape) == (6, 6), f"data.x 모양은 (6, 6) 이어야 해요. 지금은 {tuple(data.x.shape)} 예요"
assert tuple(data.edge_index.shape) == (2, 14), f"edge_index 모양은 (2, 14) 여야 해요. 지금은 {tuple(data.edge_index.shape)} 예요. 두 줄, 칸 14개"
assert data.edge_index.dtype == torch.long, "edge_index 는 dtype=torch.long 이어야 해요"
_pairs = set(zip(data.edge_index[0].tolist(), data.edge_index[1].tolist()))
_want = set(_edges) | {(_v, _u) for _u, _v in _edges}
assert _pairs == _want, f"엣지가 달라요. 빠지거나 더 들어간 것: {sorted(_pairs ^ _want)}"
assert data.num_nodes == 6 and data.num_edges == 14, "노드 6개, 엣지 14개여야 해요"
assert torch.equal(data.y, _y6), "data.y 는 [0, 0, 0, 1, 1, 1] 이어야 해요"
assert hasattr(data, 'train_mask') and torch.equal(data.train_mask, _mask6), "train_mask 는 0번과 5번만 True 여야 해요"
print("통과! 2주차 PracticeCode_2 셀 8 의 `simple_graph = Data(x=x, edge_index=edge_index)` 와 셀 10 의 `data.train_mask` 가 이 모양이에요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.data import Data

# 가짜 그래프: 0-1-2 삼각형, 3-4-5 삼각형, 2-3 다리
edge_index = torch.tensor([[0, 1, 0, 2, 1, 2, 2, 3, 3, 4, 3, 5, 4, 5],
                           [1, 0, 2, 0, 2, 1, 3, 2, 4, 3, 5, 3, 5, 4]], dtype=torch.long)
x = torch.eye(6)
y = torch.tensor([0, 0, 0, 1, 1, 1])
data = Data(x=x, edge_index=edge_index, y=y)
data.train_mask = torch.zeros(6, dtype=torch.bool)
data.train_mask[0] = True
data.train_mask[5] = True
print(data, data.num_nodes, data.num_edges)
```

</details>

## 6. 빈칸: GCNConv 로 2층 GCN 모델  (빈칸 채우기)

직접 짠 GCNLayer 대신 PyG 가 준 GCNConv 로 2층 모델 PyGGCN 을 완성해요.

- `GCNConv(입력 수, 출력 수)` 는 c12-07 GCNLayer 와 같은 일을 해요. 속에서 자기 자신 선 더하기, 차수 -1/2 제곱 정규화, 가중치 곱하기를 하고 bias 도 더해요.
- 다른 점은 A_hat 대신 **edge_index 를 받는다** 는 것이에요. 그래서 `self.conv1(x, edge_index)` 처럼 불러요.
- `nn.Dropout(p=0.5)` 는 학습 때 숨은 특징 칸을 무작위로 절반 0 으로 만들어 외우기만 하는 것을 막아요. `model.eval()` 이면 꺼져요(c11).
- `return h, logits` 는 두 값을 함께 돌려줘요(c04 여러 값 return).
- 빈칸 3개: 첫 층 출력 수, 둘째 층 입력 수, 첫 층에 넘길 엣지.

<details><summary>힌트 1</summary>

c12-09 ScratchGCN 빈칸과 같은 자리예요

</details>

<details><summary>힌트 2</summary>

GCNConv 는 A_hat 대신 무엇을 받죠?

</details>

<details><summary>힌트 3</summary>

hidden_dim, hidden_dim, edge_index

</details>


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

edge_index = torch.tensor([[0, 1, 0, 2, 1, 2, 2, 3, 3, 4, 3, 5, 4, 5],
                           [1, 0, 2, 0, 2, 1, 3, 2, 4, 3, 5, 3, 5, 4]], dtype=torch.long)
data = Data(x=torch.eye(6), edge_index=edge_index, y=torch.tensor([0, 0, 0, 1, 1, 1]))

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, ___)
        self.conv2 = GCNConv(___, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, ___)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

torch.manual_seed(42)
model = PyGGCN(data.num_node_features, 2)
model.eval()
logits = model(data.x, data.edge_index)
print(logits.shape)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
from torch_geometric.nn import GCNConv as _GCNConv
assert 'PyGGCN' in globals() and issubclass(PyGGCN, nn.Module), "PyGGCN 클래스가 있어야 해요"
assert 'logits' in globals() and tuple(logits.shape) == (6, 2), "logits 모양은 (노드 6, 반 2) = (6, 2) 여야 해요"
_m = PyGGCN(6, 3, hidden_dim=10)
assert isinstance(_m.conv1, _GCNConv) and isinstance(_m.conv2, _GCNConv), "conv1, conv2 는 GCNConv 여야 해요"
assert tuple(_m.conv1.lin.weight.shape) == (10, 6), f"PyGGCN(6, 3, hidden_dim=10) 의 conv1 은 6 → 10 이어야 해요. 지금 weight 모양 {tuple(_m.conv1.lin.weight.shape)}. 첫 빈칸은 hidden_dim"
assert tuple(_m.conv2.lin.weight.shape) == (3, 10), f"conv2 는 10 → 3 이어야 해요. 지금 weight 모양 {tuple(_m.conv2.lin.weight.shape)}. 둘째 빈칸도 hidden_dim"
_ei = torch.tensor([[0, 1, 0, 2, 1, 2, 2, 3, 3, 4, 3, 5, 4, 5], [1, 0, 2, 0, 2, 1, 3, 2, 4, 3, 5, 3, 5, 4]])
_m.eval()
torch.manual_seed(5)
_x = torch.randn(6, 6)
with torch.no_grad():
    _h = F.relu(_m.conv1(_x, _ei))
    _want = _m.conv2(_h, _ei)
    _hid, _out = _m(_x, _ei, return_hidden=True)
assert tuple(_hid.shape) == (6, 10), f"return_hidden=True 면 숨은 특징 모양 (6, 10) 이 먼저 나와야 해요. 지금은 {tuple(_hid.shape)} 예요"
assert torch.allclose(_out, _want, atol=1e-5), "결과가 달라요. 첫 층에도 edge_index 를 넘겨요"
print("통과! 2주차 PracticeCode_2 셀 66 의 `PyGGCN` 이 이 클래스예요. 셀 60 KarateClubGNN 도 GCNConv 자리만 CustomGCNConv 로 바꾼 같은 모양이에요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

edge_index = torch.tensor([[0, 1, 0, 2, 1, 2, 2, 3, 3, 4, 3, 5, 4, 5],
                           [1, 0, 2, 0, 2, 1, 3, 2, 4, 3, 5, 3, 5, 4]], dtype=torch.long)
data = Data(x=torch.eye(6), edge_index=edge_index, y=torch.tensor([0, 0, 0, 1, 1, 1]))

class PyGGCN(nn.Module):
    '''Same 2-layer architecture, but using PyG GCNConv.'''

    def __init__(self, num_features, num_classes, hidden_dim=16):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x, edge_index, return_hidden=False):
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = self.dropout(h)
        logits = self.conv2(h, edge_index)
        if return_hidden:
            return h, logits
        return logits

torch.manual_seed(42)
model = PyGGCN(data.num_node_features, 2)
model.eval()
logits = model(data.x, data.edge_index)
print(logits.shape)
```

</details>

## 7. 고치기: edge_index 를 (엣지 수, 2) 로 만들었어요  (고치기)

엣지를 (출발, 도착) 짝의 리스트로 적어 텐서로 만들었더니 GCNConv 에서 에러가 나요. edge_index 를 2줄짜리 (2, 14) 모양으로 고쳐요.

- 짝 리스트를 그대로 `torch.tensor` 에 넣으면 짝 하나가 한 줄이 되어 모양이 (14, 2) 예요. PyG 는 **위 줄 출발, 아래 줄 도착** 인 (2, 14) 를 기대해요(c08).
- `.t()` 는 줄과 칸을 뒤집어요(transpose). numpy 의 `A.T` 와 같아요(c07). (14, 2) 가 (2, 14) 가 돼요.
- `.contiguous()` 는 뒤집은 텐서를 메모리에 가지런히 다시 놓아요. PyG 예제에서 `.t().contiguous()` 로 붙여 쓰는 일이 많아요.
- 에러 메시지에 edge_index 나 size 라는 말이 보이면 모양부터 print 해 봐요.

<details><summary>힌트 1</summary>

print(edge_index.shape) 를 해 보면 (14, 2) 예요

</details>

<details><summary>힌트 2</summary>

줄과 칸을 뒤집는 메서드는 .t()

</details>

<details><summary>힌트 3</summary>

edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()

</details>


In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

pairs = [[0, 1], [1, 0], [0, 2], [2, 0], [1, 2], [2, 1], [2, 3],
         [3, 2], [3, 4], [4, 3], [3, 5], [5, 3], [4, 5], [5, 4]]
edge_index = torch.tensor(pairs, dtype=torch.long)
data = Data(x=torch.eye(6), edge_index=edge_index)
torch.manual_seed(0)
conv = GCNConv(6, 2)
out = conv(data.x, data.edge_index)
print(data.edge_index.shape, out.shape)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
assert 'data' in globals() and 'out' in globals(), "data 와 out 이 있어야 해요"
assert tuple(data.edge_index.shape) == (2, 14), f"edge_index 모양은 (2, 14) 여야 해요. 지금은 {tuple(data.edge_index.shape)} 예요. .t() 로 뒤집어요"
_pairs = set(zip(data.edge_index[0].tolist(), data.edge_index[1].tolist()))
_want = set(_edges) | {(_v, _u) for _u, _v in _edges}
assert _pairs == _want, "위 줄은 출발, 아래 줄은 도착이어야 해요. 짝 리스트를 .t() 로 뒤집었는지 봐요"
assert tuple(out.shape) == (6, 2), f"out 모양은 (노드 6, 출력 2) 여야 해요. 지금은 {tuple(out.shape)} 예요"
with torch.no_grad():
    _Wt = conv.lin.weight
    _want_out = _Ahat6 @ (data.x @ _Wt.T) + conv.bias
assert torch.allclose(out.detach(), _want_out, atol=1e-5), "GCNConv 결과가 A_hat @ (x @ W.T) + bias 와 달라요. 엣지가 맞는지 봐요"
print("통과! 채점에서 GCNConv 결과가 c12-06 방식의 A_hat @ (x @ W.T) + bias 와 똑같은지 확인했어요. GCNConv 속이 정말 GCN 한 층이에요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

pairs = [[0, 1], [1, 0], [0, 2], [2, 0], [1, 2], [2, 1], [2, 3],
         [3, 2], [3, 4], [4, 3], [3, 5], [5, 3], [4, 5], [5, 4]]
edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
data = Data(x=torch.eye(6), edge_index=edge_index)
torch.manual_seed(0)
conv = GCNConv(6, 2)
out = conv(data.x, data.edge_index)
print(data.edge_index.shape, out.shape)
```

</details>

## 8. 직접 짜기: 노트북의 GCNLayer 와 ScratchGCN  (직접 짜기)

2주차 PracticeCode_2.ipynb 셀 55 의 두 클래스 GCNLayer 와 ScratchGCN 을 주석 단계만 보고 처음부터 다시 짜요.

- 원본: 2주차 PracticeCode_2.ipynb 셀 55. 원본은 가라테 클럽 그래프로 모델을 만들지만, 채점은 노드 6개짜리 가짜 그래프(0-1-2 삼각형, 3-4-5 삼각형, 2-3 다리)로 해요.
- 채점은 `torch.manual_seed(42)` 로 씨앗을 같게 두고 원본 클래스와 내 클래스의 결과가 똑같은지 비교해요. 층을 만드는 순서(conv1 먼저, conv2 다음)가 같아야 가중치 숫자도 같아요.
- 클래스 첫 줄의 `'''...'''` 는 설명 문자열(docstring)이에요. 실행에는 영향이 없지만 원본처럼 적어 봐요.
- c12-07 과 c12-09 를 합친 것이에요. 막히면 그 두 실습을 다시 봐요.

<details><summary>힌트 1</summary>

c12-07 의 GCNLayer 를 먼저 옮기고, c12-09 의 ScratchGCN 을 붙여요

</details>

<details><summary>힌트 2</summary>

두 클래스 모두 __init__ 첫 줄은 super().__init__()

</details>

<details><summary>힌트 3</summary>

ScratchGCN forward: h = self.conv1(x, A_hat, activation=True) 다음 if return_embeddings: return h

</details>

<details><summary>힌트 4</summary>

마지막 줄: return self.conv2(h, A_hat, activation=False)

</details>


In [ ]:
import torch
import torch.nn as nn

# 2주차 PracticeCode_2.ipynb 셀 55 의 두 클래스를 다시 짜요
# 1. class GCNLayer(nn.Module):
#    - 설명 문자열 '''One layer: H' = activation(A_hat H W).'''
#    - __init__(self, in_dim, out_dim): super().__init__() 다음 self.W = nn.Linear(in_dim, out_dim, bias=False)
#    - forward(self, H, A_hat, activation=True): out = A_hat @ self.W(H)
#      activation 이 참이면 torch.relu(out), 아니면 out 을 return (한 줄 if)
# 2. class ScratchGCN(nn.Module):
#    - 설명 문자열 '''Two-layer GCN.'''
#    - __init__(self, in_dim, hidden_dim, num_classes): super().__init__() 다음
#      self.conv1 = GCNLayer(in_dim, hidden_dim), self.conv2 = GCNLayer(hidden_dim, num_classes)
#    - forward(self, x, A_hat, return_embeddings=False):
#      h = self.conv1(x, A_hat, activation=True)
#      return_embeddings 가 참이면 h 를 return
#      아니면 self.conv2(h, A_hat, activation=False) 를 return
class GCNLayer(nn.Module):
    ...

class ScratchGCN(nn.Module):
    ...

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
assert 'GCNLayer' in globals() and issubclass(GCNLayer, nn.Module), "GCNLayer 는 nn.Module 을 물려받아야 해요"
assert 'ScratchGCN' in globals() and issubclass(ScratchGCN, nn.Module), "ScratchGCN 은 nn.Module 을 물려받아야 해요"
try:
    torch.manual_seed(42)
    _m = ScratchGCN(6, 16, 2)
except TypeError as _e:
    raise AssertionError(f"ScratchGCN(6, 16, 2) 를 만들 수 없어요: {_e}. __init__(self, in_dim, hidden_dim, num_classes) 를 만들어요")
assert hasattr(_m, 'conv1') and isinstance(_m.conv1, GCNLayer), "self.conv1 = GCNLayer(in_dim, hidden_dim) 이 있어야 해요"
assert hasattr(_m, 'conv2') and isinstance(_m.conv2, GCNLayer), "self.conv2 = GCNLayer(hidden_dim, num_classes) 가 있어야 해요"
assert isinstance(getattr(_m.conv1, 'W', None), nn.Linear) and _m.conv1.W.bias is None, "GCNLayer 에 self.W = nn.Linear(in_dim, out_dim, bias=False) 가 있어야 해요"
_names = [n for n, _ in _m.named_parameters()]
assert _names == ['conv1.W.weight', 'conv2.W.weight'], f"학습 숫자 이름은 conv1.W.weight, conv2.W.weight 두 개여야 해요. 지금은 {_names} 예요"
torch.manual_seed(42)
_r = _ScratchGCN(6, 16, 2)
with torch.no_grad():
    _out = _m(_X6, _Ahat6)
    _emb = _m(_X6, _Ahat6, return_embeddings=True)
    _rout = _r(_X6, _Ahat6)
    _remb = _r(_X6, _Ahat6, return_embeddings=True)
assert _out is not None and tuple(_out.shape) == (6, 2), "model(x, A_hat) 결과는 (노드 6, 반 2) 모양이어야 해요. forward 에 return 이 있는지 봐요"
assert torch.allclose(_emb, _remb, atol=1e-6), "return_embeddings=True 결과가 원본과 달라요. 첫 층은 activation=True 이고 A_hat @ self.W(H) 순서예요"
assert torch.allclose(_out, _rout, atol=1e-6), "모델 결과가 원본과 달라요. 마지막 층은 activation=False 예요"
torch.manual_seed(0)
_m2 = ScratchGCN(6, 4, 3)
torch.manual_seed(0)
_r2 = _ScratchGCN(6, 4, 3)
with torch.no_grad():
    assert torch.allclose(_m2(_X6, _Ahat6), _r2(_X6, _Ahat6), atol=1e-6), "ScratchGCN(6, 4, 3) 에서 결과가 달라요. 숫자 대신 hidden_dim, num_classes 이름을 썼는지 봐요"
print("통과! 수업 노트북의 GCN 모델 클래스를 혼자 짰어요. 노트북 셀 55 를 열어 한 줄씩 비교해 봐요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
```

</details>

## 9. 직접 짜기: 노트북의 학습 함수 train_dense_model  (직접 짜기)

2주차 PracticeCode_2.ipynb 셀 48 의 accuracy_from_logits 와 train_dense_model 을 주석 단계만 보고 다시 짜고, c12-13 의 ScratchGCN 을 가짜 그래프에서 학습시켜요.

- 원본: 2주차 PracticeCode_2.ipynb 셀 48 (함수 두 개)과 셀 55 끝의 학습 부르기. 원본은 가라테 클럽 34명과 300바퀴지만, 여기서는 **노드 6개짜리 가짜 그래프** 와 100바퀴로 바꿨어요. A_hat 은 c12-06 방식으로 만들어 두었어요.
- 학습 루프 줄들은 c11 과 같아요: train(), zero_grad(), 예측, train_mask 노드만 손실, backward(), step(). 그다음 eval() 과 no_grad() 로 정확도를 재요.
- `weight_decay=5e-4` 는 가중치 숫자가 너무 커지지 않게 조금씩 줄여 주는 설정이에요. 5e-4 는 0.0005 예요.
- `_, train_acc, full_acc = ...` 의 `_` 는 "이 값은 안 쓸게요" 라는 뜻의 이름이에요. pred 를 버려요.
- `epoch % 20 == 0` 은 20 으로 나눈 나머지가 0, 곧 20바퀴마다라는 뜻이에요(c01 의 %).
- 채점은 씨앗 42 로 원본 함수와 내 함수의 손실 기록이 같은지, 손실이 줄어드는지 봐요. print 모양은 채점하지 않아요.

<details><summary>힌트 1</summary>

c11-13 train 함수와 c11-14 accuracy 함수를 합친 모양이에요

</details>

<details><summary>힌트 2</summary>

for epoch in range(1, epochs + 1): 안에 학습 여섯 줄, 그다음 eval 과 no_grad

</details>

<details><summary>힌트 3</summary>

loss = F.cross_entropy(logits[train_mask], y[train_mask])

</details>

<details><summary>힌트 4</summary>

마지막은 return history_loss, history_train, history_full

</details>


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


# 가짜 그래프: 0-1-2 삼각형, 3-4-5 삼각형, 2-3 다리 (노트북의 가라테 클럽 대신)
edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
A = torch.zeros(6, 6)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(6)
D_inv_sqrt = torch.diag(A_tilde.sum(dim=1).pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

X = torch.eye(6)
y = torch.tensor([0, 0, 0, 1, 1, 1])
train_mask = torch.zeros(6, dtype=torch.bool)
train_mask[0] = True
train_mask[5] = True

# 2주차 PracticeCode_2.ipynb 셀 48 의 두 함수를 다시 짜요
# 1. def accuracy_from_logits(logits, y, train_mask):
#    - pred = logits.argmax(dim=1)
#    - train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
#    - full_acc = (pred == y).float().mean().item()
#    - pred, train_acc, full_acc 를 return
# 2. def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
#    - optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
#    - history_loss, history_train, history_full = [], [], []
#    - print(f"Training {name}") 와 print("=" * 60)
#    - for epoch in range(1, epochs + 1):
#        model.train(), optimizer.zero_grad(), logits = model(x, support)
#        loss = F.cross_entropy(logits[train_mask], y[train_mask]), loss.backward(), optimizer.step()
#        model.eval() 다음 with torch.no_grad(): 안에서
#            logits_eval = model(x, support)
#            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)
#        세 리스트에 loss.item(), train_acc, full_acc 를 append
#        epoch % 20 == 0 or epoch == 1 이면 epoch, loss, train acc, full-graph acc 를 print
#    - print("=" * 60) 후 세 리스트를 return
# 3. torch.manual_seed(42) 다음 scratch_gcn = ScratchGCN(6, hidden_dim=16, num_classes=2)
#    gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(scratch_gcn, X, A_hat, y, train_mask, epochs=100, name="the scratch GCN")
def accuracy_from_logits(logits, y, train_mask):
    ...


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    ...

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
_edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
def _norm_t(A):
    n = A.shape[0]
    At = A + torch.eye(n)
    D = torch.diag(At.sum(dim=1).pow(-0.5))
    return D @ At @ D
_A6 = torch.zeros(6, 6)
for _u, _v in _edges:
    _A6[_u, _v] = 1.0
    _A6[_v, _u] = 1.0
_Ahat6 = _norm_t(_A6)
_X6 = torch.eye(6)
_y6 = torch.tensor([0, 0, 0, 1, 1, 1])
_mask6 = torch.zeros(6, dtype=torch.bool)
_mask6[0] = True
_mask6[5] = True
class _GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
class _ScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _GCNLayer(in_dim, hidden_dim)
        self.conv2 = _GCNLayer(hidden_dim, num_classes)
    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)
def _acc(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    return pred, (pred[train_mask] == y[train_mask]).float().mean().item(), (pred == y).float().mean().item()
def _train(model, x, support, y, train_mask, epochs=300, lr=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for epoch in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        loss = F.cross_entropy(lg[train_mask], y[train_mask])
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            _, ta, fa = _acc(model(x, support), y, train_mask)
        hl.append(loss.item()); ht.append(ta); hf.append(fa)
    return hl, ht, hf
assert 'accuracy_from_logits' in globals() and callable(accuracy_from_logits), "accuracy_from_logits 함수를 만들어야 해요"
_lg = torch.tensor([[2.0, 1.0], [0.0, 3.0], [1.0, 0.5], [0.2, 0.1], [0.0, 1.0], [1.0, 2.0]])
_ra = accuracy_from_logits(_lg, _y6, _mask6)
assert _ra is not None and len(_ra) == 3, "accuracy_from_logits 는 pred, train_acc, full_acc 세 값을 return 해야 해요"
_p, _ta, _fa = _ra
assert torch.equal(_p, torch.tensor([0, 1, 0, 0, 1, 1])), f"pred 가 달라요. 행마다 argmax(dim=1) 예요. 지금은 {_p.tolist()} 예요"
assert isinstance(_ta, float) and abs(_ta - 1.0) < 1e-6, f"train_acc 는 train_mask 노드(0번, 5번)만 보면 1.0 이에요. 지금은 {_ta} 예요. .item() 으로 숫자를 꺼냈는지 봐요"
assert isinstance(_fa, float) and abs(_fa - 4 / 6) < 1e-6, f"full_acc 는 6개 중 4개 맞아 0.6667 이에요. 지금은 {_fa} 예요"
assert 'train_dense_model' in globals() and callable(train_dense_model), "train_dense_model 함수를 만들어야 해요"
torch.manual_seed(42)
_m = _ScratchGCN(6, 16, 2)
_res = train_dense_model(_m, _X6, _Ahat6, _y6, _mask6, epochs=40, lr=0.05, name="check")
assert _res is not None and len(_res) == 3, "train_dense_model 은 history_loss, history_train, history_full 세 리스트를 return 해야 해요"
_hl, _ht, _hf = _res
assert len(_hl) == 40 and len(_ht) == 40 and len(_hf) == 40, f"epochs=40 이면 리스트마다 40개여야 해요. 지금은 {len(_hl)}, {len(_ht)}, {len(_hf)} 개예요. range(1, epochs + 1) 과 append 들여쓰기를 봐요"
assert all(isinstance(v, float) for v in _hl), "history_loss 에는 loss.item() 숫자를 넣어요"
torch.manual_seed(42)
_r = _ScratchGCN(6, 16, 2)
_rl, _rt, _rf = _train(_r, _X6, _Ahat6, _y6, _mask6, epochs=40, lr=0.05)
assert abs(_hl[0] - _rl[0]) < 1e-5, f"첫 바퀴 손실이 원본과 달라요. 기준 {_rl[0]:.4f}, 지금 {_hl[0]:.4f}. train_mask 노드만 cross_entropy 했는지 봐요"
assert all(abs(a - b) < 1e-4 for a, b in zip(_hl, _rl)), f"손실 기록이 원본과 달라요. 기준 마지막 {_rl[-1]:.4f}, 지금 {_hl[-1]:.4f}. zero_grad, weight_decay=5e-4, lr 을 확인해요"
assert _hl[-1] < _hl[0], f"학습하면 손실이 줄어야 해요. 처음 {_hl[0]:.4f}, 마지막 {_hl[-1]:.4f}"
assert all(abs(a - b) < 1e-6 for a, b in zip(_hf, _rf)), "전체 정확도 기록이 원본과 달라요. model.eval() 과 no_grad 안에서 다시 예측해 accuracy_from_logits 로 계산해요"
assert _ht[-1] == 1.0, "학습이 끝나면 정답 아는 노드 0번, 5번은 다 맞혀야 해요"
assert _m.training is False, "바퀴마다 정확도를 잴 때 model.eval() 로 바꿔야 해요"
print("통과! 수업 노트북의 GCN 모델과 학습 함수를 모두 직접 짰어요. 노트북에서 Karate Club 데이터로 바꾸면 셀 55 와 똑같이 돌아요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


# 가짜 그래프: 0-1-2 삼각형, 3-4-5 삼각형, 2-3 다리 (노트북의 가라테 클럽 대신)
edges = [(0, 1), (0, 2), (1, 2), (2, 3), (3, 4), (3, 5), (4, 5)]
A = torch.zeros(6, 6)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(6)
D_inv_sqrt = torch.diag(A_tilde.sum(dim=1).pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

X = torch.eye(6)
y = torch.tensor([0, 0, 0, 1, 1, 1])
train_mask = torch.zeros(6, dtype=torch.bool)
train_mask[0] = True
train_mask[5] = True

def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, support)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_eval = model(x, support)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d}/{epochs} | loss={loss.item():.4f} "
                f"| train acc={train_acc:.4f} | full-graph acc={full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full


torch.manual_seed(42)
scratch_gcn = ScratchGCN(6, hidden_dim=16, num_classes=2)
gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(
    scratch_gcn, X, A_hat, y, train_mask, epochs=100, name="the scratch GCN"
)
```

</details>